In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import csv
import os
from datetime import datetime
from pathlib import Path
import time
from nominal_lib import *
from getCSV import makeCSV
from getIV import getIvVd, getIvVg
from scipy.stats import qmc
import re

#SPICE_FILE = "nfet_base.spice"
PARENT_DIR = Path.home() / "ngspice-skywater-sims/montecarlo/mc_output_lhc"
desired_device = 0

In [ ]:
N_RUNS = 10000

In [ ]:
device_library = [("nmos_FET_len_0p15_wid_1p6", model_devs[0],'n', '0'),
           ("nmos_FET_len_0p19_wid_7", model_devs[1],'n', '1'),
           ("nmos_FET_len_0p25_wid_1p6", model_devs[2],'n','2'),
           ('nmos_FET_len_1_wid_1p6', model_devs[3],'n','3'),
           ('nmos_FET_len_1_wid_3', model_devs[4],'n', '4'),
           ('nmos_FET_len_8_wid_1p6', model_devs[5],'n', '5'),
           ('nmos_FET_len_20_wid_0p64', model_devs[6],'n','6'),
           ('nmos_FET_len_100_wid_100', model_devs[7],'n','7'),
           ('pmos_FET_len_8_wid_0p84', model_devs[8],'p','8'),
           ('pmos_FET_len_0p35_wid_1p6', model_devs[9],'p','0'),
           ('pmos_FET_len_0p5_wid_0p42', model_devs[10],'p','7'),
           ('pmos_FET_len_0p35_wid_0p55', model_devs[11],'p','5'),
           ('pmos_FET_len_0p5_wid_0p64', model_devs[12],'p','4'),
           ('pmos_FET_len_0p35_wid_5', model_devs[13],'p','9'),
           ('pmos_FET_len_2_wid_5', model_devs[14],'p','11'),
           ('pmos_FET_len_4_wid_7', model_devs[15],'p','2'),
           ('pmos_FET_len_8_wid_1p6', model_devs[16],'p','6'),
           ('pmos_FET_len_8_wid_5', model_devs[17],'p','1')
]

In [23]:
#getting appropriate parameter sweeping range
import re
import shutil
import subprocess
import tempfile
import os
import sys
def getParameterRange(circuit, fetType):
    #if fetType == 
    NGSPICE_BIN = os.path.expanduser('~/.conda/envs/spice_env/bin/ngspice')
    print(NGSPICE_BIN)
    LIB_PATH = ("/home/oliviag/skywater130nm/volare/sky130/versions/"
                "a918dc7c8e474a99b68c85eb3546b4ed91fe9e7b/sky130A/"
                "libs.tech/ngspice/sky130.lib.spice")
    CORNER = "tt_77k"
    alters = """\ """
    body = """\ """
    TEMP_K = 77.0
    TEMP_C = TEMP_K - 273.15
    PARAMS = ["vth0", "u0", "rdsw", "nfactor", "vsat", "eta0", "delta"]
    ONE = {p: 1.0 for p in PARAMS}
    
    #VGS_SWEEP = [round(0.05 * i, 3) for i in range(19)]

    temp_c = TEMP_C
    #Search windows based on behavior estimations
    TOL       = 1e-3                   
    MAX_ITERS = 40
    DEBUG     = False
    searchLow = { "vth0" : 1e-10,
                  "u0" : 1e-10,
                  "rdsw" : 1e-10,
                  "nfactor" : 1e-10,
                  "vsat" : 1e-10,
                  "eta0" : 1e-10,
                  "delta" : 1e-10
    }
    searchHigh = {"vth0" : 5,
                  "u0" : 5,
                  "rdsw" : 5,
                  "nfactor" : 50,
                  "vsat" : 5,
                  "eta0" : 5,
                  "delta" : 10
    }
     

    NETLIST_TMPL = f"""\
    BSIM4 77K multiplier test
    {circuit}    
    .lib "{LIB_PATH}" {CORNER}
    .temp {temp_c}
    
    
    
    vd d 0 0.9
    vg g 0 0.9
    
    .options bsim4_ignore_checklimit=0
    
    .control
    op
    quit
    .endc
    .end
    """
    DECK = NETLIST_TMPL
    BAD = ["fatal", "simulation(s) aborted", "singular matrix", "no convergence",
           "iteration limit reached", "error on line", "no such parameter",
           "unknown subckt", "could not find", "can't find"]
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def read_base_value(param, text):
        m = re.search(rf"\b{re.escape(param)}\s*=\s*([-+0-9.eE]+)", text)
        if not m:
            raise ValueError(f"Could not find '{param}' in model file")
        return float(m.group(1))
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def build_model_text(base_text, overrides):
        text = base_text
        for param, value in overrides.items():
            pattern = rf"(\b{re.escape(param)}\s*=\s*)([-+0-9.eE]+)"
            if not re.search(pattern, text):
                raise ValueError(f"Param '{param}' not found in model card to override")
            text = re.sub(pattern, lambda m: f"{m.group(1)}{value:.10g}", text, count=1)
        return text
        #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def build_deck(mults, vg, body):
        alters = "".join(f"alterparam {p}_nom = {mults[p]:.10g}\n" for p in PARAMS)
        return DECK.format(tk=TEMP_K, lib=LIB_PATH, corner=CORNER,
                           tc=TEMP_C, vg=vg, alters=alters, body=body)
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def _run(text, d, fname="test.cir"):
        path = os.path.join(d, fname)
        with open(path, "w") as f:
            f.write(text)
        try:
            pr = subprocess.run([NGSPICE_BIN, "-b", path],
                                capture_output=True, text=True, timeout=30)
        except subprocess.TimeoutExpired:
            return False, "TIMEOUT"
        out = pr.stdout + pr.stderr
        if DEBUG:
            print(f"--- ngspice rc={pr.returncode} ---\n{out}\n")
        ok = pr.returncode == 0 and not any(b in out.lower() for b in BAD)
        return ok, out
    ONE = {p: 1.0 for p in PARAMS}
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def run_ngspice(model_text, base_dir):
        """Write a modified model file + netlist, run ngspice -b, return (ok, output)."""
        model_path = os.path.join(base_dir, "model_under_test.spice")
        with open(model_path, "w") as f:
            f.write(model_text)
    
        netlist_path = os.path.join(base_dir, "test.cir")
        with open(netlist_path, "w") as f:
            f.write(NETLIST_TMPL.format(
                model_path=model_path,
                temp_c=TEMP_C,
                model_name=MODEL_NAME,
                
            ))
    
        try:
            proc = subprocess.run(
                [NGSPICE_BIN, "-b", netlist_path],
                capture_output=True, text=True, timeout=30,
            )
        except subprocess.TimeoutExpired:
            return False, "TIMEOUT"
    
        output = proc.stdout + proc.stderr
        ok = (proc.returncode == 0) and ("fatal" not in output.lower())
        if DEBUG:
            print(f"--- ngspice run (ok={ok}) ---\n{output}\n")
        return ok, output
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def test_mults(mults):
        return run_op(mults)[0]
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def test_multiplier(param, mult):
        return test_mults({p: (mult if p == param else 1.0) for p in PARAMS})
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def bisect_bound(param, good, bad):
        for _ in range(MAX_ITERS):
            if abs(good - bad) < TOL:
                break
            mid = (good + bad) / 2.0
            if test_multiplier(param, mid):
                good = mid
            else:
                bad = mid
        return good
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def find_bounds(param):
        #print(param)
        HI_START = searchHigh[param]
        LO_START = searchLow[param]
        if test_multiplier(param, HI_START):
            upper, unote = HI_START, f">= {HI_START} (no failure in bracket)"
        else:
            upper = bisect_bound(param, 1.0, HI_START)
            unote = f"{upper:.4f}"
        if test_multiplier(param, LO_START):
            lower, lnote = LO_START, f"<= {LO_START} (no failure in bracket)"
        else:
            lower = bisect_bound(param, 1.0, LO_START)
            lnote = f"{lower:.4f}"
        return lower, upper, lnote, unote
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def test_combo(overrides_mult, base_values, base_text):
        overrides = {p: base_values[p] * overrides_mult[p] for p in PARAMS}
        model_text = build_model_text(base_text, overrides)
        with tempfile.TemporaryDirectory() as d:
            return run_ngspice(model_text, d)
    VGS_SWEEP = [round(0.05 * i, 3) for i in range(19)]   # 0 -> 0.90 V
    IV_NETLIST_TMPL = f"""\
    BSIM4 77K IV sweep
    {circuit}
    .lib "{LIB_PATH}" {CORNER}
    .temp {temp_c}
    
    
    
    vd d 0 0.9
    vg g 0 0
    
    
    .control
    {alters}reset
    {body}
    quit
    .endc
    .end
    """
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def run_iv(mults):
        with tempfile.TemporaryDirectory() as d:
            csv = os.path.join(d, "iv.csv")
            body = f"dc vg 0 0.9 0.05\nwrdata {csv} i(vd)"
            ok, out = _run(build_deck(mults, "0", body), d, "iv.cir")
            if not ok or not os.path.isfile(csv):
                return False, None
            curve = []
            with open(csv) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 2:
                        try:
                            curve.append((float(parts[0]), -float(parts[1])))
                        except ValueError:
                            continue
        return (len(curve) > 1), (curve or None)
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def run_op(mults):
        with tempfile.TemporaryDirectory() as d:
            return _run(build_deck(mults, "0.9", "op"), d)
    #//////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
    def run_iv_sweep(model_text, base_dir):
        """Run a full Id-Vgs sweep, return (ok, list_of_(vgs,id)) or (False, None)."""
        model_path = os.path.join(base_dir, "model_under_test.spice")
        with open(model_path, "w") as f:
            f.write(model_text)
    
        out_csv = os.path.join(base_dir, "iv.csv")
        netlist_path = os.path.join(base_dir, "iv.cir")
        with open(netlist_path, "w") as f:
            f.write(IV_NETLIST_TMPL.format(
                model_path=model_path, temp_c=TEMP_C, model_name=MODEL_NAME,
                 out_csv=out_csv,
            ))
    
        try:
            proc = subprocess.run(
                [NGSPICE_BIN, "-b", netlist_path],
                capture_output=True, text=True, timeout=30,
            )
        except subprocess.TimeoutExpired:
            return False, None
    
        output = proc.stdout + proc.stderr
        ok = (proc.returncode == 0) and ("fatal" not in output.lower()) and os.path.isfile(out_csv)
        if not ok:
            return False, None
    
        curve = []
        with open(out_csv) as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 2:
                    try:
                        curve.append((float(parts[0]), float(parts[1])))
                    except ValueError:
                        continue
        return True, curve
    def getRange():
        print(f"Target temp: {TEMP_K} K ({TEMP_C:.2f} C)")
    
        ok, out = run_op(ONE)
        print(f"Baseline (all _nom = 1) at {TEMP_K}K: {'PASS' if ok else 'FAIL'}")
        if not ok:
            print(out)
            sys.exit("Baseline fails before any multiplier is applied -- fix setup first.")
    
        _, c1 = run_iv(ONE)
        _, c2 = run_iv({**ONE, "u0": 0.5})
        if c1 and c2 and c1 == c2:
            sys.exit("u0_nom = 0.5 gave an identical curve -- alterparam is a no-op. "
                     "Check the _nom hooks in the model card.")
        print("alterparam hooks confirmed live (u0_nom changed the IV curve).")
    
        print(f"\n{'param':10s} {'min mult':>14s} {'max mult':>14s}")
        print("-" * 42)
        results = {}
        for p in PARAMS:
            lower, upper, lnote, unote = find_bounds(p)
            results[p] = (lower, upper)
            print(f"{p:10s} {lnote:>14s} {unote:>14s}")
    
        print("\nNOTE: per-parameter bounds with the other 6 held at 1.0. A joint")
        print("draw can still fail (esp. u0/vsat/rdsw interactions).")
        return results
    ranges = getRange()
    #print(ranges)
    return(ranges)
    


In [21]:
#circuit = 'XM1 DRAIN GATE 0 0 sky130_fd_pr__nfet_01v8_lvt l=0.15 w=1.6 mult=1 m=1'

#checkedRanges = getParameterRange(circuit)

/home/oliviag/.conda/envs/spice_env/bin/ngspice
Target temp: 77.0 K (-196.15 C)
Baseline (all _nom = 1) at 77.0K: PASS
alterparam hooks confirmed live (u0_nom changed the IV curve).

param            min mult       max mult
------------------------------------------
vth0       <= 1e-10 (no failure in bracket) >= 5 (no failure in bracket)
u0         <= 1e-10 (no failure in bracket) >= 5 (no failure in bracket)
rdsw       <= 1e-10 (no failure in bracket) >= 5 (no failure in bracket)


KeyboardInterrupt: 

In [ ]:
def getHypercube(num_samples, runDir, circuit_ng, fetType):
    num_vars = 7 
    PARAMS = ["vth0", "vsat", "rdsw", "u0", "nfactor", "eta0", "delta"]
    
    checkedRanges = getParameterRange(circuit_ng, fetType)
    # 2. Your specified multipliers/ranges, calculated using paramRangeValidation
    n_bounds = {
        "vth0": checkedRanges['vth0'],
        "vsat": checkedRanges['vsat'],
        "rdsw": checkedRanges['rdsw']
        "u0": checkedRanges['u0'],
        "nfactor": checkedRanges['nfactor'],
        "eta0": checkedRanges['eta0'],
        "delta": checkedRanges['delta'],
    }
    
    
    #num_samples = 500
    sampler = qmc.LatinHypercube(d=num_vars)
    sample_matrix = sampler.random(n=num_samples)
    
    #floors and ceilings in the exact order of PARAMS
    floor = np.array([n_bounds[p][0] for p in PARAMS])
    ceiling = np.array([n_bounds[p][1] for p in PARAMS])
    
    #scale 0 to 1 custom bounds
    scaled_matrix = qmc.scale(sample_matrix, floor, ceiling)
    
    '''np.savetxt(
        os.path.join(runDir, "lhs_parameters.txt"), scaled_matrix, fmt="%.16e"
    )
    print(f"Generated LHS parameter matrix of shape: {scaled_matrix.shape}")'''
        
    # Export to a text file for ngspice to read
    np.savetxt(os.path.join(runDir,'lhs_parameters.txt'), scaled_matrix, fmt='%.16e')
   
    return

In [ ]:
def lhcVisual(runDir):
    labels = ["vth0", "vsat", "rdsw", "u0", "nfactor", "eta0", "delta"]
    
    file_path = os.path.join(runDir, 'lhs_parameters.txt')
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return
        
    matrix = np.loadtxt(file_path)
    num_samples, num_vars = matrix.shape
    fig, axes = plt.subplots(num_vars, num_vars, figsize=(14, 14), sharex='col')
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
    
    for i in range(num_vars):
        for j in range(num_vars):
            ax = axes[i, j]       
            #corner grid layout
            if j > i:
                ax.set_visible(False)
                continue    
            #histogram on diagonal
            if i == j:
                ax.hist(matrix[:, i], bins=12, color=colors[i], edgecolor='white', alpha=0.8)
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
            #plotting multivariable scatter representations
            else:
                ax.scatter(matrix[:, j], matrix[:, i], color='#555555', alpha=0.5, edgecolors='none', s=20)
                ax.set_ylim(matrix[:, i].min() * 0.98, matrix[:, i].max() * 1.02)
            #boundaries and layout frames
            ax.set_xlim(matrix[:, j].min() * 0.98, matrix[:, j].max() * 1.02)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            #format peripheral label text on edge borders only
            if j == 0 and i != 0:
                ax.set_ylabel(labels[i], fontsize=11, fontweight='bold')
            if i == num_vars - 1:
                ax.set_xlabel(labels[j], fontsize=11, fontweight='bold')
                ax.tick_params(axis='x', labelrotation=45)
    #saving
    fig.tight_layout()
    outdir = os.path.join(runDir, "allPlots")
    outfile = os.path.join(outdir, "LHC_Sampling_Visualization.png")
    fig.savefig(outfile, dpi=300, bbox_inches="tight")
    
    print(f"Saved {outfile}")
    plt.close(fig)
    return

In [ ]:
def deviceTemplate(dev_dir, dev_param, devType):
    if devType == 'p':
        biasV = -1.48
    else:
        biasV = 1.48
    home = str(Path.home())
    #nfet_meas = 
    
    device_cmd = f"""* NMOS sweep (Sky130) 


{dev_param} nf=1 ad='floor((nf+1)/2) * w/nf * 0.29' as='floor((nf+2)/2) * w/nf * 0.29' pd='2 * floor((nf+1)/2) * (w/nf +0.29)' ps='2 * floor((nf+2)/2) * (w/nf + 0.29)' nrd='0.29 / w' nrs='0.29 / w' sa=0 sb=0 sd=0 

.lib "{home}/skywater130nm/volare/sky130/versions/a918dc7c8e474a99b68c85eb3546b4ed91fe9e7b/sky130A/libs.tech/ngspice/sky130.lib.spice" tt_77k

Vgs gate GND {biasV}
Vds VDD GND {biasV}
.temp -196.15


"""
    file = 'mc_template.spice'
    outFile = os.path.join(dev_dir, file)
    if not os.path.exists(outFile):
        with open(outFile, "w+") as f:
            f.write(device_cmd)
            f.flush()
        
    return outFile

In [ ]:
#setting up data file
start_time = time.perf_counter()
folder = datetime.now().strftime(f"{N_RUNS}_runs_data_%Y-%m-%d_%H-%M-%S")
print(foler)

In [ ]:
#setting up chosen device for sweep
device = device_library[desired_device]
deviceName = device[0]
circuit = device[1]
fet = device[2]
nom_params_key = f"{fet}fet_mod_{device[3]}"
modP = mods_list.get(nom_params_key, {})

In [ ]:
# directory + template setup
dev_dir = PARENT_DIR / deviceName
run_dir = dev_dir / folder
run_dir.mkdir(parents=True, exist_ok=True)
print(run_dir)
plotDir = run_dir / "allPlots"
plotDir.mkdir(parents=True, exist_ok=True)
deviceTemplate(dev_dir, circuit, fet)
SPICE_FILE = os.path.join(dev_dir, "mc_template.spice")

In [ ]:
# boundary setup and LHC run (Generates all parameters at once)
'''param_init = np.array([
    float(modP["vth0_nom"]), 
    float(modP["vsat_nom"]), 
    float(modP["rdsw_nom"]), 
    float(modP["u0_nom"]), 
    float(modP["nfactor_nom"]), 
    float(modP["eta0_nom"]), 
    float(modP["delta_nom"])
])'''


In [1]:
getHypercube(N_RUNS, run_dir, circuit, fet)
lhs_runs = np.loadtxt(os.path.join(run_dir, "lhs_parameters.txt"))
lhcVisual(run_dir)

NameError: name 'getHypercube' is not defined

In [ ]:
# Base template configuration
base_template = open(SPICE_FILE).read()
logFile = os.path.join(run_dir, "mc_run.log")

CHUNK_SIZE = 10000 #running max 10k curves per netlist
total_runs = len(lhs_runs)

In [ ]:
# Process genertes IV for each generated param set in increments of 10,000
for chunk_start in range(0, total_runs, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_runs)
    chunk_runs = lhs_runs[chunk_start:chunk_end]
    
    control_lines = [
        ".control",
        "set noaskquit",
        "set filetype=ascii",
        "run",
    ]
    
    # Build control lines keeping the absolute global index (idx) for file naming
    for offset, run in enumerate(chunk_runs):
        idx = chunk_start + offset
        v_vth0, v_vsat, v_rdsw, v_u0, v_nfactor, v_eta0, v_delta = run
        runPath = Path(run_dir)
        outPath = runPath
        vds_file = outPath / f"vds_sweep_{idx}.dat"
        vgs_file = outPath / f"vgs_sweep_{idx}.dat"

        if fet == 'n':
            control_lines.extend([
                f'echo " MONTE CARLO RUN: {idx}"',
                f'alterparam vth0_nom = {v_vth0}',
                f'alterparam u0_nom = {v_u0}',
                f'alterparam rdsw_nom = {v_rdsw}',
                f'alterparam nfactor_nom = {v_nfactor}',
                f'alterparam vsat_nom = {v_vsat}',
                f'alterparam eta0_nom = {v_eta0}',
                f'alterparam delta_nom = {v_delta}',
                'reset',
                #'alter Vgs 1.48',
                #'dc Vds 0 1.85 0.01',
                #f'let nom_vth0    = {v_vth0}',
                #f'let nom_u0      = {v_u0}',
                #f'let nom_rdsw    = {v_rdsw}',
                #f'let nom_nfactor = {v_nfactor}',
                #f'let nom_vsat    = {v_vsat}',
                #f'let nom_eta0    = {v_eta0}',
                #f'let nom_delta   = {v_delta}',
                #f'let run_num     = {idx}',
                #'let ax_id0 = -vds#branch',
                #'set wr_singlescale',
                #f'wrdata {vds_file} vdd gate ax_id0 nom_vth0 nom_u0 nom_rdsw nom_nfactor nom_vsat nom_eta0 nom_delta',
                'dc Vgs 0 1.85 0.01',
                f'let nom_vth0    = {v_vth0}',
                f'let nom_u0      = {v_u0}',
                f'let nom_rdsw    = {v_rdsw}',
                f'let nom_nfactor = {v_nfactor}',
                f'let nom_vsat    = {v_vsat}',
                f'let nom_eta0    = {v_eta0}',
                f'let nom_delta   = {v_delta}',
                f'let run_num     = {idx}',
                'let ax_id2 = -vds#branch',
                'set wr_singlescale',
                f'wrdata {vgs_file} vdd gate ax_id2 nom_vth0 nom_u0 nom_rdsw nom_nfactor nom_vsat nom_eta0 nom_delta'
            ])
        else:
            control_lines.extend([
                f'echo " MONTE CARLO RUN: {idx}"',
                f'alterparam vth0_nom = {v_vth0}',
                f'alterparam u0_nom = {v_u0}',
                f'alterparam rdsw_nom = {v_rdsw}',
                f'alterparam nfactor_nom = {v_nfactor}',
                f'alterparam vsat_nom = {v_vsat}',
                f'alterparam eta0_nom = {v_eta0}',
                f'alterparam delta_nom = {v_delta}',
                'dc Vds 0 -1.85 -0.01',
                f'let nom_vth0    = {v_vth0} * (v(gate)*0 + 1)',
                f'let nom_u0      = {v_u0} * (v(gate)*0 + 1)',
                f'let nom_rdsw    = {v_rdsw} * (v(gate)*0 + 1)',
                f'let nom_nfactor = {v_nfactor} * (v(gate)*0 + 1)',
                f'let nom_vsat    = {v_vsat} * (v(gate)*0 + 1)',
                f'let nom_eta0    = {v_eta0} * (v(gate)*0 + 1)',
                f'let nom_delta   = {v_delta} * (v(gate)*0 + 1)',
                f'let run_num     = {idx} * (v(gate)*0 + 1)',
                'let ax_id0 = -vds#branch',
                'set wr_singlescale',
                f'wrdata {vds_file} vdd gate ax_id0 nom_vth0 nom_u0 nom_rdsw nom_nfactor nom_vsat nom_eta0 nom_delta',
                'dc Vgs 0 -1.85 -0.01',
                f'let nom_vth0    = {v_vth0} * (v(gate)*0 + 1)',
                f'let nom_u0      = {v_u0} * (v(gate)*0 + 1)',
                f'let nom_rdsw    = {v_rdsw} * (v(gate)*0 + 1)',
                f'let nom_nfactor = {v_nfactor} * (v(gate)*0 + 1)',
                f'let nom_vsat    = {v_vsat} * (v(gate)*0 + 1)',
                f'let nom_eta0    = {v_eta0} * (v(gate)*0 + 1)',
                f'let nom_delta   = {v_delta} * (v(gate)*0 + 1)',
                f'let run_num     = {idx} * (v(gate)*0 + 1)',
                'let ax_id2 = -vds#branch',
                f'wrdata {vgs_file} vdd gate ax_id2 nom_vth0 nom_u0 nom_rdsw nom_nfactor nom_vsat nom_eta0 nom_delta',
                'reset'
            ])

    control_lines.extend(["quit", ".endc", ".end"])

    # Write chunk-specific spice deck
    with open("mc_run.spice", "w+") as f:
        f.write(base_template)
        f.write("\n\n")
        f.write("\n".join(control_lines))
        f.flush()

    # Execute ngspice for this batch
    mode = 'a' if chunk_start > 0 else 'w'
    with open(logFile, mode) as log:
        subprocess.run(["ngspice", "-b", "-q", "mc_run.spice"], stdout=log, stderr=subprocess.STDOUT)

    print(f"Finished sweeps for runs {chunk_start} to {chunk_end - 1}.")
end_time = time.perf_counter()


In [ ]:
duration = end_time - start_time
print(f"Task completed in {duration:.4f} seconds.")